# Classical Models

In [12]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

## Data setup

In [13]:
clinical = pd.read_csv('../data/Demo_data.csv')

X = clinical.drop(columns = ['Patient ID', 'Steatosis stage', 'Fibroscan S', 'Fibroscan F'])
Y = clinical[['Steatosis stage']]

encoder = OneHotEncoder(drop = 'if_binary', sparse_output = False)
X[['sex']] = encoder.fit_transform(X[['sex']])

In [14]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.25, random_state = 1234)

## Train with 4 classes (S0, S1, S2, S3)

In [15]:
def trainRandomForest(x_train, x_test, y_train, y_test):
    rf = RandomForestClassifier(class_weight = 'balanced')

    param_grid = {
        'n_estimators': [20, 50, 100, 125, 150, 175, 200],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 7],
    }

    kf = KFold(
        n_splits = 10,
        shuffle = True,
        random_state = 1234,
    )

    grid_search = GridSearchCV(
        estimator = rf,
        param_grid = param_grid,
        cv = kf,
        scoring = 'f1_macro',
        n_jobs = -1,
    )

    grid_search.fit(x_train, y_train.values.ravel())

    print(f'Best Params: {grid_search.best_params_}')
    print(f'Best Score: {grid_search.best_score_:.4f}')

    best_model = grid_search.best_estimator_
    test_score = best_model.score(x_test, y_test)
    print(f'Test Score: {test_score}')

trainRandomForest(x_train, x_test, y_train, y_test)

Best Params: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 175}
Best Score: 0.5640
Test Score: 0.5172413793103449


In [16]:
def trainXGB(x_train, x_test, y_train, y_test):
    xgb = XGBClassifier()

    param_grid = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [20, 50, 100, 125, 150, 175, 200],
        'subsample': [0.8, 1.0]
    }

    kf = KFold(
        n_splits = 10,
        shuffle = True,
        random_state = 1234,
    )

    grid_search = GridSearchCV(
        estimator = xgb,
        param_grid = param_grid,
        cv = kf,
        scoring = 'f1_macro',
        n_jobs = -1,
    )

    grid_search.fit(x_train, y_train.values.ravel())

    print(f'Best Params: {grid_search.best_params_}')
    print(f'Best Score: {grid_search.best_score_:.4f}')

    best_model = grid_search.best_estimator_
    test_score = best_model.score(x_test, y_test)
    print(f'Test Score: {test_score}')

trainXGB(x_train, x_test, y_train, y_test)

Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 150, 'subsample': 1.0}
Best Score: 0.5555
Test Score: 0.41379310344827586


Models did not perform well. I believe the high class imbalance (seen in EDA.ipynb).

## Train with 2 classes (is greater than s0)

In [17]:
binary_split = train_test_split(X, Y > 1, test_size = 0.25, random_state = 1234)

In [18]:
trainRandomForest(*binary_split)

Best Params: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 125}
Best Score: 0.7396
Test Score: 0.6896551724137931


In [19]:
trainXGB(*binary_split)


Best Params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 175, 'subsample': 0.8}
Best Score: 0.8027
Test Score: 0.6551724137931034


Training with a binary class target led to better scores, though still not great.

## Synthetic Minority Over-sampling Technique (SMOTE)

In [ ]:
# TODO: Look into SMOTE and tabular oversampling

# Something to keep in mind once images are used, implement patient-stratified data split